# Laboratorio: espacios afines y suma directa

Este laboratorio convierte los criterios teóricos en cálculos exactos con
`sympy`. El objetivo no es confiar ciegamente en una función, sino interpretar
rangos, núcleos y sistemas de ecuaciones.

Al terminar podrás:

- describir el conjunto solución compatible como $x_p+\ker(A)$;
- calcular una base de $U+W$ y la dimensión de $U\cap W$;
- decidir si una suma es directa;
- obtener explícitamente una base de la intersección.


In [ ]:
import sympy as sp
from sympy import Matrix
sp.init_printing()

def matriz_columnas(vectores):
    return Matrix.hstack(*(Matrix(v) for v in vectores))

def datos_suma(U, W):
    # U y W tienen como columnas bases de dos subespacios de R^n.
    if U.rows != W.rows:
        raise ValueError("Los subespacios deben pertenecer al mismo espacio ambiente.")
    if U.rank() != U.cols or W.rank() != W.cols:
        raise ValueError("Las columnas de entrada deben ser bases (familias LI).")
    B = U.row_join(W)
    dim_interseccion = U.cols + W.cols - B.rank()
    return {
        "dim_U": U.cols,
        "dim_W": W.cols,
        "dim_suma": B.rank(),
        "dim_interseccion": dim_interseccion,
        "suma_directa": dim_interseccion == 0,
        "genera_ambiente": B.rank() == U.rows,
        "base_suma": Matrix.hstack(*B.columnspace()),
    }

def base_interseccion(U, W):
    # Base de Col(U) intersección Col(W), con U y W de rango columna completo.
    K = U.row_join(-W)
    candidatos = []
    for z in K.nullspace():
        a = z[:U.cols, :]
        candidatos.append(U * a)
    if not candidatos:
        return sp.zeros(U.rows, 0)
    C = Matrix.hstack(*candidatos)
    return Matrix.hstack(*C.columnspace())


## 1. Un sistema compatible es un conjunto afín

Para $Ax=b$, una solución particular $x_p$ fija el punto de apoyo y
$\ker(A)$ contiene todas las direcciones. Verificaremos

$$\{x:Ax=b\}=x_p+\ker(A).$$


In [ ]:
A = Matrix([[1, 2, 1],
            [2, 4, 3],
            [1, 2, 2]])
b = Matrix([4, 10, 6])

solucion_general = sp.linsolve((A, b))
base_nucleo = A.nullspace()
x_p = Matrix([2, 0, 2])

display(solucion_general)
display(x_p)
display(base_nucleo)
assert A * x_p == b
assert all(A * z == sp.zeros(A.rows, 1) for z in base_nucleo)


El resultado es

$$
x=\begin{bmatrix}2\\0\\2\end{bmatrix}
t\begin{bmatrix}-2\\1\\0\end{bmatrix}.
$$

El conjunto no es subespacio porque no contiene al vector cero, pero es una
recta afín paralela al núcleo.


## 2. Suma directa que completa el espacio ambiente

Tomemos el plano $U=\operatorname{span}\{e_1,e_2\}$ y el eje
$W=\operatorname{span}\{e_3\}$ en $\mathbb R^3$.


In [ ]:
U = matriz_columnas([(1, 0, 0), (0, 1, 0)])
W = matriz_columnas([(0, 0, 1)])

info = datos_suma(U, W)
info


In [ ]:
v = Matrix([3, -2, 5])
B = U.row_join(W)
coef = B.LUsolve(v)
u = U * coef[:U.cols, :]
w = W * coef[U.cols:, :]

display(coef)
display(u, w)
assert u + w == v


Como las columnas reunidas forman una base de $\mathbb R^3$, los
coeficientes —y por tanto la descomposición $v=u+w$— son únicos.


## 3. Suma que no es directa

Si dos subespacios comparten una dirección no nula, la representación deja de
ser única.


In [ ]:
U = matriz_columnas([(1, 0, 0), (0, 1, 0)])
W = matriz_columnas([(1, 1, 0), (0, 0, 1)])

info = datos_suma(U, W)
I = base_interseccion(U, W)
display(info)
display(I)

assert I.rank() == info["dim_interseccion"]
assert U.row_join(I).rank() == U.rank()
assert W.row_join(I).rank() == W.rank()


La intersección es la recta generada por $(1,1,0)$. Por ejemplo,
$0=0+0=(1,1,0)+(-1,-1,0)$ da dos descomposiciones distintas en $U+W$.


## 4. Caso histórico en $\mathbb R^4$

Este ejemplo procede de materiales anteriores del curso. Determinaremos si
$U\oplus W=\mathbb R^4$ sin convertirlo todavía en una lista de evaluación.


In [ ]:
U = matriz_columnas([(1, 2, 0, 1), (0, 1, 1, 1)])
W = matriz_columnas([(1, 0, 1, -1), (1, 1, 1, 0)])

info = datos_suma(U, W)
I = base_interseccion(U, W)
display(U.row_join(W))
display(info)
display(I)


El rango reunido es $3$, no $4$, y la intersección tiene dimensión
$1$. Por tanto, este par histórico **no** satisface
$U\oplus W=\mathbb R^4$. La conclusión se obtiene del cálculo, no del hecho de
que cada subespacio tenga dos generadores.


## 5. Para explorar

1. Modifica una columna de $W$ para que la suma del caso anterior sea directa.
2. Construye dos planos distintos de $\mathbb R^3$. Antes de calcular, predice
   la dimensión mínima posible de su intersección.
3. Cambia $b$ en el primer ejemplo por un vector incompatible. ¿Por qué ya no
   existe un punto $x_p$ y, por tanto, tampoco una traslación del núcleo?
4. Explica por qué el criterio de rango funciona solo como está escrito cuando
   las columnas de entrada son bases.
